# Figure 2 · Honest evaluation framework: why random splits overestimate

> One figure, one notebook. Main panels read sequences from `00_data` Full-set artifacts; Cluster OOD uses fixed **k=12** (`_common.CLUSTER_K`) to define leave-one-cluster-out splits in ESM embedding space. All panels use real data.

Core claim (DataSAIL idea on real data): **cross-split sequence similarity = information leakage**. Under random splits, test sequences more easily find near-identical train neighbors (high leakage). Patient, Cluster, and Subtype OOD are complementary axes: they isolate different structure, so leakage need not be strictly monotone, but together they show random splits are overly optimistic.

- **2A｜Cross-split nearest-neighbour similarity** — under Random / Patient / Cluster / Subtype, distribution of each test sequence's nearest-neighbour similarity to train.
- **2B｜Near-duplicate leakage rate** — per split and drug class, fraction of test sequences with `similarity ≥ 0.99`.
- **2C｜Leakage vs apparent AUROC** — link near-duplicate leakage rate to binary-baseline AUROC to show that random splits pair high leakage with more optimistic AUROC.

> Naming aligned with DataSAIL: random=R, patient grouping=I1, sequence-isolation idea=S1; this paper's Cluster OOD uses fixed k=12. Citation: Joeres et al., Nat Commun 2025.


In [1]:
from __future__ import annotations

from pathlib import Path

import json
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "work" / "prepared").exists() else CWD.parent
assert (PROJECT_ROOT / "work" / "prepared").exists(), f"cannot locate work/prepared from {CWD}"

PREP = PROJECT_ROOT / "work" / "prepared"
EMB = PROJECT_ROOT / "work" / "emb_full"
RES = PROJECT_ROOT / "work" / "results"
FIG_DIR = PROJECT_ROOT / "figures" / "manuscript" / "fig2"
OUT_DIR = PROJECT_ROOT / "results" / "notebooks" / "fig2"
# Cluster OOD k fixed contract; CSV no longer required for main pipeline.
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SURFACE, INK, INK2, INK3, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1"
SPLIT_COLOR = {"random": "#d03b3b", "patient": "#2a78d6", "cluster": "#9467bd", "subtype": "#1baf7a"}
SPLIT_LABEL = {"random": "Random", "patient": "Patient", "cluster": "Cluster", "subtype": "Subtype"}
MAIN_CLASSES = ["PI", "NRTI", "NNRTI", "INI"]
GENE_OF = {"PI": "PR", "NRTI": "RT", "NNRTI": "RT", "INI": "IN"}
K_STAR = 12
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 11, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": False,
    "font.family": "DejaVu Sans",
})
print("fig2 setup ready | selected cluster k =", K_STAR, "|", sorted(p.name for p in PREP.glob("*.csv")))

fig2 setup ready | selected cluster k = 12 | ['CAI.csv', 'INI.csv', 'NNRTI.csv', 'NRTI.csv', 'PI.csv']


In [ ]:
# Load precomputed cross-split sequence-similarity stats (from notebooks/scripts/fig2_cross_split_similarity.py)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "scripts"))
from fig2_cross_split_similarity import load_dist

leak = pd.read_csv(OUT_DIR / "cross_split_similarity.csv")
_dist_path = OUT_DIR / "cross_split_dist.npz"
dist = load_dist(_dist_path) if _dist_path.exists() else {}
if not dist:
    print("WARNING: cross_split_dist.npz not found. Run fig2_cross_split_similarity.py first.")
print(f"Loaded cross_split_similarity.csv: {leak.shape} | dist arrays: {len(dist)}")
print(leak.to_string(index=False))


## 2A · Cross-split nearest-neighbour similarity

For each test sequence, compute its **nearest-neighbour normalized Hamming similarity** to the train set (= fraction of sites identical to the most similar training sequence). Values closer to 1 mean near-duplicates of the test sequence are easy to find in train. Four splits side by side: Random / Patient / Cluster / Subtype.


In [ ]:
# 2A main: 4 classes × 4 splits nearest-neighbour similarity distributions
fig, axes = plt.subplots(1, len(MAIN_CLASSES), figsize=(14.5, 4.4), sharey=True)
schemes = ["random", "patient", "cluster", "subtype"]
short_labels = {"random": "Random", "patient": "Patient", "cluster": "Cluster", "subtype": "Subtype"}
for ax, cls in zip(axes, MAIN_CLASSES):
    data, cols, xt = [], [], []
    for sc in schemes:
        if (cls, sc) in dist:
            data.append(dist[(cls, sc)]); cols.append(SPLIT_COLOR[sc]); xt.append(sc)
    parts = ax.violinplot(data, positions=range(len(data)), showmedians=True, widths=0.75)
    for b, c in zip(parts["bodies"], cols):
        b.set_facecolor(c); b.set_alpha(0.55); b.set_edgecolor(c)
    for key in ("cmedians", "cbars", "cmins", "cmaxes"):
        if key in parts:
            parts[key].set_color(INK2); parts[key].set_linewidth(1.0)
    ax.set_xticks(range(len(xt)))
    ax.set_xticklabels([short_labels[s] for s in xt], fontsize=9, rotation=25, ha="right")
    ax.set_title(cls, color=INK, fontsize=11, pad=6)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylabel("Nearest-neighbour similarity")
axes[0].set_ylim(0.80, 1.005)
fig.suptitle("Fig 2A - Cross-split nearest-neighbour similarity",
             x=0.5, y=1.02, fontsize=12, color=INK, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2A_crosssplit_similarity.png", dpi=150, bbox_inches="tight")
print("saved fig2A_crosssplit_similarity.png")
plt.show()

## 2B · Near-duplicate leakage rate

Compress the 2A distribution to one scalar: fraction of test sequences with `similarity ≥ 0.99`. Four splits side-by-side per class; higher values mean more near-duplicates of test sequences exist in train.

In [ ]:
# 2B main: near-duplicate test fraction (frac ≥0.99), class × split grouped bars
schemes = ["random", "patient", "cluster", "subtype"]
fig, ax = plt.subplots(figsize=(10.2, 4.8))
x = np.arange(len(MAIN_CLASSES)); w = 0.20
for j, sc in enumerate(schemes):
    vals = [leak[(leak.drug_class == c) & (leak.scheme == sc)]["frac_ge_0.99"].values
            for c in MAIN_CLASSES]
    vals = [v[0] * 100 if len(v) else np.nan for v in vals]
    ax.bar(x + (j - 1.5) * w, vals, w, color=SPLIT_COLOR[sc], edgecolor=SURFACE,
           label=SPLIT_LABEL[sc])
    for xi, v in zip(x + (j - 1.5) * w, vals):
        if not np.isnan(v):
            ax.text(xi, v + 1, f"{v:.0f}", ha="center", va="bottom", fontsize=8, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(MAIN_CLASSES, fontsize=10)
ax.set_ylabel("Near-duplicate test sequences (%)")
ax.set_title("Fig 2B - Near-duplicate leakage rate (similarity ≥ 0.99)",
             loc="left", color=INK, fontsize=11.5, pad=10)
ax.legend(frameon=False, fontsize=8.5, loc="upper left", ncol=2)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2B_leakage_magnitude.png", dpi=150, bbox_inches="tight")
print("saved fig2B_leakage_magnitude.png")
plt.show()

## 2C · Leakage vs apparent AUROC

Link the `similarity ≥ 0.99` near-duplicate leakage rate to binary-mutation-baseline AUROC. Each point is one evaluation protocol; the point is that Random CV has both the highest leakage and more optimistic AUROC, while Patient / Cluster / Subtype are complementary stricter axes (not required to be strictly monotone among themselves).

In [ ]:
# 2C：near-duplicate leakage rate vs binary baseline AUROC
# Load precomputed results (from notebooks/scripts/fig2_cross_split_similarity.py)
link = pd.read_csv(OUT_DIR / "leakage_vs_binary_auc.csv")
print("Protocol source: leakage_vs_binary_auc.csv")
print(link[["scheme", "leak_pct", "auroc"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6.8, 5.0))
for _, r in link.iterrows():
    ax.scatter(r["leak_pct"], r["auroc"], s=150, color=SPLIT_COLOR[r["scheme"]],
               edgecolor=SURFACE, linewidth=1.6, zorder=3)
    ax.text(r["leak_pct"] + 0.8, r["auroc"], r["label"], ha="left", va="center",
            fontsize=9.5, color=INK)
valid = link.dropna(subset=["leak_pct", "auroc"])
if len(valid) >= 3:
    coef = np.polyfit(valid["leak_pct"], valid["auroc"], 1)
    xs = np.linspace(valid["leak_pct"].min(), valid["leak_pct"].max(), 100)
    ax.plot(xs, np.polyval(coef, xs), "--", color=INK2, lw=1.4, alpha=0.65)
    corr = np.corrcoef(valid["leak_pct"], valid["auroc"])[0, 1]
    ax.text(0.02, 0.04, f"Pearson r={corr:+.2f}", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=9.5, color=INK2)
ax.set_xlabel("Near-duplicate test sequences (%)")
ax.set_ylabel("Mean AUROC (binary baseline)")
ax.set_title("Fig 2C - Leakage and apparent AUROC by evaluation protocol",
             loc="left", color=INK, fontsize=11.5, pad=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2C_leakage_vs_auc.png", dpi=150, bbox_inches="tight")
print("saved fig2C_leakage_vs_auc.png")
plt.show()

## Combined Figure 2 · panels A–C

Redraw A–C from Fig2 intermediate tables and in-memory variables instead of reading single-panel PNGs, so fonts, sizes, line widths, and colors stay consistent.

In [ ]:
# Combined Figure 2: redraw A-C from data/tables, with a unified visual system
TEXT_COLOR = "#000000"
PANEL_TITLE_SIZE = 13
AXIS_LABEL_SIZE = 12
TICK_LABEL_SIZE = 10
LEGEND_TEXT_SIZE = 9
ANNOTATION_SIZE = 9
schemes = ["random", "patient", "cluster", "subtype"]
short_labels = {"random": "Random", "patient": "Patient", "cluster": "Cluster", "subtype": "Subtype"}
# PI has a longer low-similarity tail; other classes are zoomed to show separation.
Y_LIMIT_BY_CLASS = {"PI": (0.80, 1.005), "NRTI": (0.92, 1.005), "NNRTI": (0.92, 1.005), "INI": (0.92, 1.005)}
LABEL_OFFSET = {"random": (0.8, 0.000), "patient": (0.8, 0.000), "cluster": (0.8, 0.000), "subtype": (0.8, -0.004)}

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": PANEL_TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,
    "legend.fontsize": LEGEND_TEXT_SIZE,
    "text.color": TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "xtick.color": TEXT_COLOR,
    "ytick.color": TEXT_COLOR,
})

fig = plt.figure(figsize=(16, 10.4), facecolor=SURFACE)
gs = fig.add_gridspec(2, 2, width_ratios=[1.35, 1.0], height_ratios=[1.0, 1.0],
                      wspace=0.26, hspace=0.42)

# ---------- A: nearest-neighbour similarity distributions ----------
gs_a = gs[0, :].subgridspec(1, len(MAIN_CLASSES), wspace=0.18)
axes_a = [fig.add_subplot(gs_a[0, i]) for i in range(len(MAIN_CLASSES))]
for ax, cls in zip(axes_a, MAIN_CLASSES):
    data, cols, xt = [], [], []
    for sc in schemes:
        if (cls, sc) in dist:
            data.append(dist[(cls, sc)]); cols.append(SPLIT_COLOR[sc]); xt.append(sc)
    parts = ax.violinplot(data, positions=range(len(data)), showmedians=True, widths=0.75)
    for b, c in zip(parts["bodies"], cols):
        b.set_facecolor(c); b.set_alpha(0.55); b.set_edgecolor(c)
    for key in ("cmedians", "cbars", "cmins", "cmaxes"):
        if key in parts:
            parts[key].set_color(INK2); parts[key].set_linewidth(1.0)
    ax.set_xticks(range(len(xt)))
    ax.set_xticklabels([short_labels[s] for s in xt], rotation=25, ha="right")
    ax.set_title(cls, fontsize=11, color=TEXT_COLOR, pad=6)
    ax.set_ylim(*Y_LIMIT_BY_CLASS[cls])
    ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE, colors=TEXT_COLOR)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes_a[0].set_ylabel("Nearest-neighbour similarity", fontsize=AXIS_LABEL_SIZE, color=TEXT_COLOR)
axes_a[0].text(-0.34, 1.12, "A  Cross-split nearest-neighbour similarity",
               transform=axes_a[0].transAxes, ha="left", va="bottom",
               fontsize=PANEL_TITLE_SIZE, fontweight="bold", color=TEXT_COLOR)

# ---------- B: near-duplicate leakage rate ----------
ax = fig.add_subplot(gs[1, 0])
x = np.arange(len(MAIN_CLASSES)); w = 0.20
for j, sc in enumerate(schemes):
    vals = [leak[(leak.drug_class == c) & (leak.scheme == sc)]["frac_ge_0.99"].values
            for c in MAIN_CLASSES]
    vals = [v[0] * 100 if len(v) else np.nan for v in vals]
    ax.bar(x + (j - 1.5) * w, vals, w, color=SPLIT_COLOR[sc], edgecolor=SURFACE,
           label=SPLIT_LABEL[sc])
    for xi, v in zip(x + (j - 1.5) * w, vals):
        if not np.isnan(v):
            ax.text(xi, v + 1, f"{v:.0f}", ha="center", va="bottom",
                    fontsize=ANNOTATION_SIZE, color=TEXT_COLOR)
ax.set_xticks(x)
ax.set_xticklabels(MAIN_CLASSES)
ax.set_ylim(0, 88)
ax.set_ylabel("Near-duplicate test sequences (%)", fontsize=AXIS_LABEL_SIZE, color=TEXT_COLOR)
ax.set_title("B  Near-duplicate leakage rate", loc="left",
             fontsize=PANEL_TITLE_SIZE, fontweight="bold", color=TEXT_COLOR, pad=8)
leg = ax.legend(frameon=False, fontsize=LEGEND_TEXT_SIZE, loc="lower left",
                bbox_to_anchor=(0.0, 0.95), ncol=4, borderaxespad=0.0)
for txt in leg.get_texts():
    txt.set_color(TEXT_COLOR)
ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE, colors=TEXT_COLOR)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

# ---------- C: leakage vs AUROC ----------
ax = fig.add_subplot(gs[1, 1])
for _, r in link.iterrows():
    ax.scatter(r["leak_pct"], r["auroc"], s=145, color=SPLIT_COLOR[r["scheme"]],
               edgecolor=SURFACE, linewidth=1.6, zorder=3)
    dx, dy = LABEL_OFFSET.get(r["scheme"], (0.8, 0.0))
    ax.text(r["leak_pct"] + dx, r["auroc"] + dy, r["label"], ha="left", va="center",
            fontsize=ANNOTATION_SIZE, color=TEXT_COLOR)
valid = link.dropna(subset=["leak_pct", "auroc"])
if len(valid) >= 3:
    coef = np.polyfit(valid["leak_pct"], valid["auroc"], 1)
    xs = np.linspace(valid["leak_pct"].min(), valid["leak_pct"].max(), 100)
    ax.plot(xs, np.polyval(coef, xs), "--", color=INK2, lw=1.4, alpha=0.65)
    corr = np.corrcoef(valid["leak_pct"], valid["auroc"])[0, 1]
    ax.text(0.03, 0.96, f"Pearson r={corr:+.2f}", transform=ax.transAxes,
            ha="left", va="top", fontsize=ANNOTATION_SIZE, color=TEXT_COLOR)
ax.set_xlim(max(0, link["leak_pct"].min() - 4), link["leak_pct"].max() + 7)
ax.set_ylim(link["auroc"].min() - 0.010, link["auroc"].max() + 0.010)
ax.set_xlabel("Near-duplicate test sequences (%)", fontsize=AXIS_LABEL_SIZE, color=TEXT_COLOR)
ax.set_ylabel("Mean AUROC (binary baseline)", fontsize=AXIS_LABEL_SIZE, color=TEXT_COLOR)
ax.set_title("C  Leakage vs apparent AUROC", loc="left",
             fontsize=PANEL_TITLE_SIZE, fontweight="bold", color=TEXT_COLOR, pad=8)
ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE, colors=TEXT_COLOR)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 1])
fig.savefig(FIG_DIR / "fig2_combined.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
fig.savefig(FIG_DIR / "fig2_combined.pdf", bbox_inches="tight", facecolor=SURFACE)
print("saved", FIG_DIR / "fig2_combined.png")
print("saved", FIG_DIR / "fig2_combined.pdf")
plt.show()